# Silver-label triage EDA

Goal: review the `silver_label` predictions **fast** by separating pairs the field evidence makes obvious from the ones a human must actually look at.

Silver labels are `det-rules OR anymatch` (see `data/silver_labels/get_silver_labels.ipynb`), so they are noisy. Here we recompute independent **field-agreement** evidence for each pair and bucket:

- **confident_match** — silver says match *and* a strong identifier agrees (SSN, or full name + DOB, or a shared phone/email with the name) and nothing strong contradicts. → safe to **bulk-accept as gold = match**.
- **confident_nonmatch** — silver says non-match *and* the names disagree / a strong ID conflicts / DOB is far off. → safe to **bulk-accept as gold = non-match**.
- **suspicious_pos** — silver says **match** but the evidence contradicts it (SSN/DOB conflict, or no shared strong key). → must review.
- **suspicious_neg** — silver says **non-match** but a strong identifier agrees (likely a missed match). → must review.
- **review** — everything genuinely ambiguous.

Output: a prioritized `silver_labels_triage_*.csv` the gold-labeling notebook consumes (bulk-accept the confident buckets, hand-review the rest).

> PHI: run on HIPAA-tier compute only. Edit paths in **Config**.

In [ ]:
# --- Bootstrap: run from anywhere, hop to the AnyMatch repo root -------------
import os, sys

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'   # Windows OpenMP guard (see CLAUDE.md)

def _find_repo_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.exists(os.path.join(d, 'loo.py')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError("Could not find repo root (no loo.py above this notebook).")
        d = parent

REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
from IPython.display import HTML, display

from utils import labeling_utils as L

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)
print('repo root:', REPO_ROOT)

## Config

In [ ]:
SILVER_CSV      = 'data/silver_labels/silver_labels_v1_2026_06_21.csv'
RECORDS_PARQUET = 'data/alliance/MDM_Population_cleaned_v3_2026_06_11.parquet'
TRIAGE_OUT      = 'silver_labels/silver_labels_triage_v1.csv'
MAX_EXAMPLES    = 6   # example pairs rendered per bucket

## 1. Load silver labels

In [ ]:
# Read silver labels. get_silver_labels.ipynb wrote with the index, so drop any
# unnamed index column and force the PATID columns to string (ID dtype trap).
silver = pd.read_csv(SILVER_CSV, dtype={'PATID_A': 'string', 'PATID_B': 'string'})
silver = silver.loc[:, ~silver.columns.str.startswith('Unnamed')]
silver['silver_label'] = (silver['silver_label'].astype('string').str.strip().str.lower()
                          .map({'true': True, 'false': False, '1': True, '0': False}))
silver = silver[['PATID_A', 'PATID_B', 'silver_label']].drop_duplicates(['PATID_A', 'PATID_B'])
print(f'{len(silver):,} silver pairs')
print(silver['silver_label'].value_counts(dropna=False))
silver.head()

## 2. Join to records

In [ ]:
# Load valid records (friendly schema) and attach <field>_a / <field>_b to every pair.
records = L.load_records(RECORDS_PARQUET)
print(f'{len(records):,} valid records; PATID index unique = {records.index.is_unique}')

pairs = L.merge_pairs(silver, records)
joinable = L.joinable_mask(pairs)
print(f'{len(pairs):,} pairs; {(~joinable).sum():,} not joinable to a valid record on one/both sides')

## 3. Per-field agreement evidence

`status_frame` gives every field one of `same / diff / one_missing / both_missing`. From those we build the boolean identity signals.

In [ ]:
status = L.status_frame(pairs)
_FALSE = pd.Series(False, index=pairs.index)

def eq(field):   s = status.get(field); return s.eq('same') if s is not None else _FALSE
def diff(field): s = status.get(field); return s.eq('diff') if s is not None else _FALSE

ssn_eq   = eq('ssn')
ssn_diff = diff('ssn')
ssn4_eq  = eq('ssn4')
ssn4_diff= diff('ssn4')
dob_eq   = eq('dob')
dob_diff = diff('dob')
first_eq = eq('first_name')
last_eq  = eq('last_name')
first_d  = diff('first_name')
last_d   = diff('last_name')
phone_eq = eq('phone')
email_eq = eq('email')

full_name_eq  = first_eq & last_eq          # first AND last agree
name_disagree = first_d & last_d            # both name anchors differ
link          = phone_eq | email_eq         # shared contact channel

# Strong, identity-grade agreement vs. contradiction.
strong_match    = ssn_eq | (full_name_eq & dob_eq) | (full_name_eq & link) | (ssn4_eq & full_name_eq)
strong_conflict = ssn_diff | (ssn4_diff & ~ssn_eq)

evid = pd.DataFrame({
    'ssn': status.get('ssn'), 'ssn4': status.get('ssn4'), 'dob': status.get('dob'),
    'first_name': status.get('first_name'), 'last_name': status.get('last_name'),
    'phone': status.get('phone'), 'email': status.get('email'),
})
print('field-status share (per field):')
for c in evid.columns:
    vc = evid[c].value_counts(normalize=True)
    print(f'  {c:11s} ' + '  '.join(f'{k}={v:.0%}' for k, v in vc.items()))

## 4. Buckets

Order matters — strongest signal assigned last so it wins. `unjoinable` pairs (a record was filtered out as invalid) can't be field-reviewed and are flagged separately.

In [ ]:
silver_true = pairs['silver_label'].fillna(False).astype(bool)

bucket = pd.Series('review', index=pairs.index, dtype=object)

# confident agreement / disagreement
bucket[silver_true & strong_match & ~strong_conflict & ~dob_diff] = 'confident_match'
bucket[~silver_true & ~strong_match & (name_disagree | strong_conflict | dob_diff)] = 'confident_nonmatch'

# silver disagrees with the field evidence -> human review
bucket[silver_true & (strong_conflict | (dob_diff & ~ssn_eq) | name_disagree)] = 'suspicious_pos'
bucket[silver_true & ~strong_match & ~strong_conflict & ~name_disagree & ~dob_diff] = 'suspicious_pos'
bucket[~silver_true & strong_match & ~strong_conflict] = 'suspicious_neg'

# can't judge from fields
bucket[~joinable] = 'unjoinable'

pairs['bucket'] = bucket
summary = (pairs.groupby('bucket')
           .agg(n=('bucket', 'size'),
                silver_match_rate=('silver_label', 'mean'))
           .sort_values('n', ascending=False))
summary['pct'] = (summary['n'] / len(pairs)).map(lambda x: f'{x:.1%}')
summary

In [ ]:
import matplotlib.pyplot as plt
order = summary.index.tolist()
colors = {'confident_match': '#2e7d32', 'confident_nonmatch': '#1565c0',
          'suspicious_pos': '#c62828', 'suspicious_neg': '#ef6c00',
          'review': '#6a1b9a', 'unjoinable': '#777777'}
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(order, summary.loc[order, 'n'], color=[colors.get(b, '#999') for b in order])
for i, b in enumerate(order):
    ax.text(i, summary.loc[b, 'n'], f"{summary.loc[b,'n']:,}", ha='center', va='bottom', fontsize=9)
ax.set(ylabel='pairs', title='Silver pairs by triage bucket')
ax.set_xticklabels(order, rotation=25, ha='right')
plt.tight_layout(); plt.show()

need_review = pairs['bucket'].isin(['suspicious_pos', 'suspicious_neg', 'review']).sum()
bulk_ok = pairs['bucket'].isin(['confident_match', 'confident_nonmatch']).sum()
print(f'bulk-acceptable : {bulk_ok:,} ({bulk_ok/len(pairs):.1%})')
print(f'need review     : {need_review:,} ({need_review/len(pairs):.1%})')

## 5. Where silver most likely disagrees with the evidence

Cross-tab of the silver label against the strongest evidence — the off-diagonal cells are the pairs worth a human's time.

In [ ]:
evidence_call = pd.Series('ambiguous', index=pairs.index, dtype=object)
evidence_call[strong_match & ~strong_conflict] = 'looks_match'
evidence_call[(name_disagree | strong_conflict | dob_diff) & ~strong_match] = 'looks_nonmatch'
ct = pd.crosstab(pairs['silver_label'], evidence_call)
print('rows = silver_label, cols = field evidence:')
ct

## 6. Example pairs per bucket

Green=equal, red=different, yellow=one missing, grey=both missing.

In [ ]:
display(HTML(L.legend_html()))

def show_bucket(name, k=MAX_EXAMPLES):
    sub = pairs[pairs['bucket'] == name]
    print(f'=== {name}: {len(sub):,} pairs (showing {min(k, len(sub))}) ===')
    html = ''.join(L.render_pair_html(r) for _, r in sub.head(k).iterrows())
    display(HTML(html))

for b in ['suspicious_neg', 'suspicious_pos', 'review', 'confident_match', 'confident_nonmatch']:
    show_bucket(b)

## 7. Export the prioritized triage queue

Written so `gold_labeling.ipynb` can bulk-accept the confident buckets and hand only the rest to the manual reviewer. `suggested_gold` is `1` / `0` for the confident buckets and blank where review is required; `priority` sorts the must-review pairs first.

In [ ]:
suggested = pd.Series(pd.NA, index=pairs.index, dtype='Int64')
suggested[pairs['bucket'] == 'confident_match'] = 1
suggested[pairs['bucket'] == 'confident_nonmatch'] = 0

priority = pairs['bucket'].map({'suspicious_neg': 0, 'suspicious_pos': 1,
                                'review': 2, 'unjoinable': 3,
                                'confident_match': 4, 'confident_nonmatch': 5}).fillna(9)

triage = pairs[['PATID_A', 'PATID_B', 'silver_label', 'bucket']].copy()
triage['suggested_gold'] = suggested
for f in ['ssn', 'ssn4', 'dob', 'first_name', 'last_name', 'phone', 'email']:
    triage[f'{f}_status'] = status.get(f)
triage['priority'] = priority.astype(int)
triage = triage.sort_values(['priority', 'PATID_A', 'PATID_B']).reset_index(drop=True)

os.makedirs(os.path.dirname(TRIAGE_OUT), exist_ok=True)
triage.to_csv(TRIAGE_OUT, index=False)
print(f'wrote {len(triage):,} rows -> {TRIAGE_OUT}')
triage.head(12)